# Results Visualization

This notebook creates a compact visualization of average HAR accuracy, FLOPs, and parameter size using summarized experiment results. It writes the generated figure to `../results`.


In [ ]:
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt

RESULTS_DIR = Path("../results")
RESULTS_DIR.mkdir(exist_ok=True)

# ---------------------------------------------------------------------
# Models & Datasets (from your SUMMARY COMPARISON tables)
# ---------------------------------------------------------------------
datasets = ["WISDM", "UCI HAR", "MotionSense", "MHEALTH", "PAMAP2"]

methods = [
    "QTModel",
    "ResNet",
    "EfficientNetB0",
    "MnasNet",
    "MobileNet",
    "MobileNetV2",
    "TSLANet",
]

# ---------------------------------------------------------------------
# Accuracies per dataset per method (rows: methods, cols: datasets)
# Using acc_mean (without the ± sd part)
# ---------------------------------------------------------------------
acc_matrix = np.array([
    [0.9315, 0.7486, 0.9107, 0.8636, 0.6500],  # QTModel
    [0.8296, 0.7483, 0.8944, 0.8157, 0.6251],  # ResNet1D
    [0.7835, 0.7429, 0.8877, 0.7104, 0.5680],  # EfficientNetB0_1D
    [0.7307, 0.7008, 0.8582, 0.6540, 0.5351],  # MnasNet_1D
    [0.7485, 0.7205, 0.8642, 0.7020, 0.5668],  # MobileNet_1D
    [0.7511, 0.7176, 0.8654, 0.6936, 0.5649],  # MobileNetV2_1D
    [0.8501, 0.6942, 0.8203, 0.7542, 0.5565],  # TSLANet
])

# ---------------------------------------------------------------------
# Params per dataset per method (from "params" column, k/M -> numbers)
# rows: methods, cols: datasets
# ---------------------------------------------------------------------
params_matrix = np.array([
    # WISDM,    UCIHAR,    MOTIONSENSE, MHEALTH,   PAMAP2
    [102.72e3, 102.72e3, 102.72e3, 103.11e3, 103.5e3],   # QTModel
    [1.50e6,   1.50e6,   1.50e6,   1.51e6,   1.51e6],    # ResNet1D
    [3.24e6,   3.24e6,   3.24e6,   3.24e6,   3.25e6],    # EfficientNetB0_1D
    [2.98e6,   2.98e6,   2.98e6,   2.99e6,   3.00e6],    # MnasNet_1D
    [3.18e6,   3.18e6,   3.18e6,   3.19e6,   3.19e6],    # MobileNet_1D
    [2.19e6,   2.19e6,   2.19e6,   2.20e6,   2.20e6],    # MobileNetV2_1D
    [314.05e3, 311.75e3, 314.05e3, 316.24e3, 316.63e3],  # TSLANet
])

# ---------------------------------------------------------------------
# FLOPs per dataset per method (MACs, from FLOPs column * 1e6)
# rows: methods, cols: datasets
# ---------------------------------------------------------------------
flops_matrix = np.array([
    # WISDM,    UCIHAR,   MOTIONSENSE, MHEALTH,   PAMAP2
    [1.89e6,   1.24e6,   1.89e6,     2.39e6,    2.39e6],   # QTModel
    [14.04e6,  9.82e6,   14.04e6,    18.04e6,   18.04e6],  # ResNet1D
    [28.29e6,  16.69e6,  28.29e6,    33.39e6,   33.40e6],  # EfficientNetB0_1D
    [24.99e6,  14.65e6,  24.99e6,    29.30e6,   29.31e6],  # MnasNet_1D
    [34.52e6,  20.86e6,  34.52e6,    41.72e6,   41.72e6],  # MobileNet_1D
    [20.18e6,  11.98e6,  20.18e6,    23.95e6,   23.96e6],  # MobileNetV2_1D
    [14.72e6,  9.32e6,   14.72e6,    18.93e6,   18.93e6],  # TSLANet
])

# ---------------------------------------------------------------------
# Average across datasets
# ---------------------------------------------------------------------
avg_acc = acc_matrix.mean(axis=1)
avg_params = params_matrix.mean(axis=1)
avg_flops = flops_matrix.mean(axis=1)

# Bubble sizes proportional to average parameters
sizes = (avg_params / avg_params.max()) * 800.0

# Colors: highlight QTModel and TSLANet, unique for others
colors = []
for m in methods:
    if m == "QTModel":
        colors.append("green")   # proposed model
    elif m == "TSLANet":
        colors.append("blue")    # strong classical baseline
    elif m == "ResNet":
        colors.append("red")
    elif m == "MobileNet":
        colors.append("orange")
    elif m == "EfficientNetB0":
        colors.append("purple")
    elif m == "MobileNetV2":
        colors.append("brown")
    elif m == "MnasNet":
        colors.append("gray")
    else:
        colors.append("black")

plt.figure()
plt.scatter(avg_flops, avg_acc, s=sizes, c=colors, alpha=0.8, edgecolors="k")

for i, m in enumerate(methods):
    plt.annotate(
        m,
        (avg_flops[i], avg_acc[i]),
        textcoords="offset points",
        xytext=(0, -12),   # shift DOWN
        ha="center",       # center under the bubble
        va="top"
    )

plt.xscale("log")
plt.xlabel("Average FLOPs across 5 HAR datasets (MACs, log scale)")
plt.ylabel("Average Accuracy across 5 HAR datasets")
plt.title("Average Accuracy vs Average FLOPs (Bubble size ∝ Average Parameters)")
plt.tight_layout()
plt.savefig(RESULTS_DIR / "har_avg_acc_vs_avg_flops_bubble_paramsize.png", dpi=300)
plt.show()
